In [1]:
import numpy as np
import pandas as pd

![Split-apply-combine paradigm for a reduction](../Images/Split-apply-combine_paradigm_for_a_reduction.png)

> **Split-apply-combine paradigm for a reduction**

![Split-apply-combine paradigm for a transformation](../Images/Split-apply-combine_paradigm_for_a_transformation.png)

> **Split-apply-combine paradigm for a transformation**

# **Group by basics**

In [2]:
df = pd.DataFrame([
    ["group_a", 0],
    ["group_a", 2],
    ["group_b", 1],
    ["group_b", 3],
    ["group_b", 5],
], columns=["group", "value"]).convert_dtypes(dtype_backend="numpy_nullable")

df

,group,value
0,group_a,0
1,group_a,2
2,group_b,1
3,group_b,3
4,group_b,5


In [4]:
df.groupby(by="group").sum()

,value
group,
group_a,2
group_b,9


In [5]:
df.groupby("group").agg("sum")

,value
group,
group_a,2
group_b,9


In [6]:
df.groupby("group").transform("sum")

,value
0,2
1,2
2,9
3,9
4,9


In [7]:
df[["value"]].div(df.groupby("group").transform("sum"))

,value
0,0.0
1,1.0
2,0.111111
3,0.333333
4,0.555556


In [8]:
df

,group,value
0,group_a,0
1,group_a,2
2,group_b,1
3,group_b,3
4,group_b,5


In [9]:
df.groupby("group", as_index=False).agg("sum")

,group,value
0,group_a,2
1,group_b,9


In [10]:
df.groupby("group").min()

,value
group,
group_a,0
group_b,1


In [13]:
df.groupby("group").agg(sum_of_value=pd.NamedAgg(column="value", aggfunc="sum"))

,sum_of_value
group,
group_a,2
group_b,9


# **Grouping and calculating multiple columns**

In [14]:
df = pd.DataFrame([
    ["North", "Widget A", "Jan", 10, 2],
    ["North", "Widget B", "Jan", 4, 0],
    ["South", "Widget A", "Jan", 8, 3],
    ["South", "Widget B", "Jan", 12, 8],
    ["North", "Widget A", "Feb", 3, 0],
    ["North", "Widget B", "Feb", 7, 0],
    ["South", "Widget A", "Feb", 11, 2],
    ["South", "Widget B", "Feb", 13, 4],
], columns=["region", "widget", "month", "sales", "returns"]).convert_dtypes(dtype_backend="numpy_nullable")

df

,region,widget,month,sales,returns
0,North,Widget A,Jan,10,2
1,North,Widget B,Jan,4,0
2,South,Widget A,Jan,8,3
3,South,Widget B,Jan,12,8
4,North,Widget A,Feb,3,0
5,North,Widget B,Feb,7,0
6,South,Widget A,Feb,11,2
7,South,Widget B,Feb,13,4


In [15]:
df.groupby("widget").sum()

,region,month,sales,returns
widget,,,,
Widget A,NorthSouthNorthSouth,JanJanFebFeb,32,7
Widget B,NorthSouthNorthSouth,JanJanFebFeb,36,12


In [16]:
df.groupby("widget")[["sales", "returns"]].agg("sum")

,sales,returns
widget,,
Widget A,32,7
Widget B,36,12


In [17]:
df.groupby("widget").agg(
    sales_total=pd.NamedAgg(column="sales", aggfunc="sum"),
    returns_total=pd.NamedAgg(column="returns", aggfunc="sum")
)

,sales_total,returns_total
widget,,
Widget A,32,7
Widget B,36,12


In [18]:
df.groupby(["widget", "region"]).agg(
    sales_total=pd.NamedAgg("sales", "sum"),
    returns_total=pd.NamedAgg("returns", "sum"),
)

sales_total  returns_total
widget   region                            
Widget A North            13              2
         South            19              5
Widget B North            11              0
         South            25             12

In [19]:
df.groupby(["widget", "region"]).agg(
    sales_total=pd.NamedAgg("sales", "sum"),
    returns_total=pd.NamedAgg("returns", "sum"),
    sales_min=pd.NamedAgg("sales", "min"),
    returns_min=pd.NamedAgg("returns", "min"),
)

sales_total  returns_total  sales_min  returns_min
widget   region                                                    
Widget A North            13              2          3            0
         South            19              5          8            2
Widget B North            11              0          4            0
         South            25             12         12            4

In [20]:
pd.Series([0, 1, 1]).mode()

0    1
dtype: int64

In [21]:
pd.Series([0, 1, 1, 2, 2]).mode()

0    1
1    2
dtype: int64

In [22]:
df = pd.DataFrame([
    ["group_a", 42],
    ["group_a", 555],
    ["group_a", 42],
    ["group_a", 555],
    ["group_b", 0],
], columns=["group", "value"])

df

,group,value
0,group_a,42
1,group_a,555
2,group_a,42
3,group_a,555
4,group_b,0


In [23]:
pd.Series([[42, 555], 0], index=pd.Index(["group_a", "group_b"]))

group_a    [42, 555]
group_b            0
dtype: object

In [24]:
pd.Series(
    [42, 555, 0],
    index=pd.Index(["group_a", "group_a", "group_b"], name="groups")
)

groups
group_a     42
group_a    555
group_b      0
dtype: int64

In [27]:
def scalar_or_list_mode(ser: pd.Series):
    result = ser.mode()
    if len(result) > 1:
        return result.tolist()
    elif len(result) == 1:
        return result.iloc[0]
    return pd.NA

def scalar_or_bust_mode(ser: pd.Series):
    result = ser.mode()
    if len(result) == 0:return pd.NA
    return result.iloc[0]

In [29]:
df.groupby("group").agg(
    scalar_or_list=pd.NamedAgg(column="value", aggfunc=scalar_or_list_mode),
    scalar_or_bust=pd.NamedAgg(column="value", aggfunc=scalar_or_bust_mode)
)

,scalar_or_list,scalar_or_bust
group,,
group_a,"[42, 555]",42
group_b,0,0


# **Group by apply**

In [31]:
df = pd.DataFrame([
    ["group_a", 42],
    ["group_a", 555],
    ["group_a", 42],
    ["group_a", 555],
    ["group_b", 0],
], columns=["group", "value"]).convert_dtypes(dtype_backend="numpy_nullable")

df

,group,value
0,group_a,42
1,group_a,555
2,group_a,42
3,group_a,555
4,group_b,0


In [34]:
pd.Series(
    [42, 555, 0],
    index=pd.Index(["group_a", "group_a", "group_b"], name="group"), 
    dtype=pd.Int64Dtype()
)

group
group_a     42
group_a    555
group_b      0
dtype: Int64

In [35]:
def mode_for_apply(df: pd.DataFrame):
    return df["value"].mode()

In [39]:
df.groupby("group").apply(mode_for_apply, include_groups=False)

group     
group_a  0     42
         1    555
group_b  0      0
Name: value, dtype: Int64

In [43]:
def mode_for_apply(df: pd.DataFrame):
    print(f"\nThe data passed to apply is:\n{df}")
    return df["value"].mode()

In [44]:
df.groupby("group").apply(mode_for_apply, include_groups=False)


The data passed to apply is:
   value
0     42
1    555
2     42
3    555

The data passed to apply is:
   value
4      0


group     
group_a  0     42
         1    555
group_b  0      0
Name: value, dtype: Int64

In [45]:
def sum_values(df: pd.DataFrame):
    return df["value"].sum()

In [46]:
df.groupby("group").apply(sum_values, include_groups=False)

group
group_a    1194
group_b       0
dtype: int64

# **Window operations**